Orginal example of LMs generation

In [1]:
import torch
import pyvene as pv

# built-in helper to get tinystore
_, tokenizer, tinystory = pv.create_gpt_neo()
emb_happy = tinystory.transformer.wte(
    torch.tensor(14628)) 

pv_tinystory = pv.IntervenableModel([{
    "layer": l,
    "component": "mlp_output",
    "intervention_type": pv.AdditionIntervention
    } for l in range(tinystory.config.num_layers)],
    model=tinystory
)
# prompt and generate
prompt = tokenizer(
    "Once upon a time there was", return_tensors="pt")
unintervened_story, intervened_story = pv_tinystory.generate(
    prompt, source_representations=emb_happy*0.3, max_length=256
)

print(tokenizer.decode(
    intervened_story[0], 
    skip_special_tokens=True
))

W0414 11:15:32.231000 27132 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


pytorch_model.bin:   0%|          | 0.00/291M [00:00<?, ?B/s]

c:\Users\vnguyen\RSML\llm-hallucinations-factual-qa\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vnguyen\.cache\huggingface\hub\models--roneneldan--TinyStories-33M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


loaded model
Once upon a time there was a little girl named Lucy. She was three years old and loved to explore. One day, Lucy was walking in the park when she saw a big, red balloon. She was so excited and wanted to play with it.

But then, a big, mean man came and said, "That balloon is mine! You can't have it!" Lucy was very sad and started to cry.

The man said, "I'm sorry, but I need the balloon for my work. You can have it if you want."

Lucy was so happy and said, "Yes please!" She took the balloon and ran away.

But then, the man said, "Wait! I have an idea. Let's make a deal. If you can guess what I'm going to give you, then you can have the balloon."

Lucy thought for a moment and then said, "I guess I'll have to get the balloon."

The man smiled and said, "That's a good guess! Here you go."

Lucy was so happy and thanked the man. She hugged the balloon and ran off to show her mom.

The end.



Trying with a truth vector

In [ ]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM


model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)

def pv_patcher(b, s): return b + s*0.1

config = pv.IntervenableConfig({
    "layer": 11,
    "component": "mlp_output",
    "intervention": pv_patcher
})

pv_model = pv.IntervenableModel(config, model=model)

qa_prompt = "In which decade did Billboard magazine first publish and American hit chart?"
prompt = tokenizer(qa_prompt, return_tensors="pt")
source_prompt = "in the 1930s" # correct answer
truth_vector = tokenizer(source_prompt, return_tensors="pt")
_, intervened_story = pv_model.generate(
    prompt, truth_vector,
    unit_locations = {"sources->base": 0},
    max_new_tokens=20
)

print(tokenizer.decode(
    intervened_story[0], 
    skip_special_tokens=True
))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In which decade did Billboard magazine first publish and American hit chart?

The answer is, in the early 1990s, the magazine was publishing a lot of hits


Trying with adding noise and scale value

In [ ]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)

qa_prompt = "In which decade did Billboard magazine first publish and American hit chart?"
base_inputs = tokenizer(qa_prompt, return_tensors="pt")

config = pv.IntervenableConfig({
    "layer": 11,
    "component": "mlp_output",
    "intervention_type": pv.AdditionIntervention
})

intervenable_model = pv.IntervenableModel(config, model=model)

noise_vector = torch.rand(model.config.n_embd) * 0.5  # random noise

target_token_index = base_inputs.input_ids.shape[1] - 1

_, intervened_outputs = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=noise_vector,
    max_new_tokens=20
)

print("Intervened Output:", tokenizer.decode(intervened_outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Intervened Output: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in


In [19]:
# unintervened
unintervened_text, _ = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=None,
    max_new_tokens=20,
    output_original_output=True
)

# add noise
_, intervened_text = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=noise_vector,
    max_new_tokens=20
)

print(f"Input prompt: {qa_prompt}")
print(f"Unintervened output:\n{tokenizer.decode(unintervened_text[0], skip_special_tokens=True)}")
print(f"Intervened output:\n{tokenizer.decode(intervened_text[0], skip_special_tokens=True)}")

print("="*60)

for scale in [0.0, 1.0, 2.0, 5.0]:
    _, output = intervenable_model.generate(
        base=base_inputs,
        unit_locations={"base": 0},
        source_representations=noise_vector * scale,
        max_new_tokens=50
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Scale {scale}: {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input prompt: In which decade did Billboard magazine first publish and American hit chart?
Unintervened output:
In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '
Intervened output:
In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 0.0: In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '80s. I think it was in the '80s. I think it was in the '80s. I think it was in the '


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 1.0: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '90s, and I think it was in the '90s, and I think it was in the '90s, and I think it was in the '90s, and I think


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Scale 2.0: In which decade did Billboard magazine first publish and American hit chart?

The answer is, in the early 1990s, the magazine was publishing a number of books, including the first two books of the Beatles' "The Beatles" and the first two albums of the Rolling Stones' "The Rolling Stones."

Scale 5.0: In which decade did Billboard magazine first publish and American hit chart?

"I think it was in the early '80s, and I think it was in the '90s," he says. "I think it was in the '90s, and I think it was in the '90s,


Trying with steering vector = correct activation - incorrect activation

In [ ]:
import torch
import pyvene as pv
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

model_dir = Path("../.cache/models/")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_dir)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=model_dir, torch_dtype=torch.bfloat16, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_qa = "In which decade did Billboard magazine first publish and American hit chart?"
pos_prompt = "It is in the early 1930s" # correct answer
neg_prompt = "It is in the early 1990s"

def get_last_layer_activation(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        return outputs.hidden_states[-1][0, -1, :].unsqueeze(0)

# Steering Vector
pos_activation = get_last_layer_activation(pos_prompt)
neg_activation = get_last_layer_activation(neg_prompt)
steering_vector = pos_activation - neg_activation

config = pv.IntervenableConfig({
    "layer": 11,  # last layer
    "component": "mlp_output",
    "intervention_type": pv.AdditionIntervention
})
intervenable_model = pv.IntervenableModel(config, model=model)

base_inputs = tokenizer(base_qa, return_tensors="pt")
target_idx = base_inputs.input_ids.shape[1]

steering_strength = 2.5 
scaled_vector = steering_vector * steering_strength

_, generated_outputs = intervenable_model.generate(
    base=base_inputs,
    unit_locations={"base": 0},
    source_representations=scaled_vector,
    max_new_tokens=20,
    # use_cache=False,
    intervene_on_prompt=True,
    do_sample=False
)

print(f"Input Prompt: {base_qa}")
print(f"Intervened Output: {tokenizer.decode(generated_outputs[0], skip_special_tokens=True)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Input Prompt: In which decade did Billboard magazine first publish and American hit chart?
Intervened Output: In which decade did Billboard magazine first publish and American hit chart?

I think it was in the early '80s. I think it was in the '
